In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

Using device: cpu


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto" if device == "cuda" else None,
)

In [ ]:
from transformers import GenerationConfig

def generate_response(prompt, system_prompt=None, max_new_tokens=512, temperature=0.3):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    outputs = chatbot(
        formatted_prompt,
        generation_config=gen_config,
    )

    full_text = outputs[0]["generated_text"]
    return full_text[len(formatted_prompt):].strip()

In [ ]:
BASELINE_SYSTEM_PROMPT = "You are a helpful assistant."

def chat_once(user_message, system_prompt=BASELINE_SYSTEM_PROMPT):
    response = generate_response(user_message, system_prompt=system_prompt)
    print("User:", user_message)
    print("\nAssistant:", response)
    return response

_ = chat_once("Patient has fever for 3 days.")

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'temperature', 'do_sample', 'pad_token_id', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to for

User: Patient has fever for 3 days.

Assistant: A patient presenting with fever for three days is an important clinical scenario that requires careful evaluation and management. Here's a step-by-step approach to consider:

### Initial Assessment

1. **History Taking:**
   - **Age, gender, and medical history:** Gather information about the patient’s age, gender, any pre-existing conditions, family history of similar symptoms, and recent travel or exposure.
   - **Fever characteristics:** Note the temperature (e.g., high-grade, low-grade), duration, pattern (steady, intermittent), and associated symptoms such as chills, sweating, headache, rash, etc.

2. **Physical Examination:**
   - **General appearance:** Look for signs of dehydration, malnutrition, or other systemic illness.
   - **Oral examination:** Check for oral ulcers, swollen lymph nodes, or other signs of infection.
   - **Skin exam:** Assess for rashes, petechiae, or other skin manifestations.
   - **Respiratory system:** Li

In [ ]:
def run_interactive_chat():
    print("MediAssist AI — type 'exit' to stop.\n")
    while True:
        user_input = input("User: ")
        if user_input.strip().lower() in ("exit", "quit"):
            print("Session ended.")
            break
        response = generate_response(user_input, system_prompt=BASELINE_SYSTEM_PROMPT)
        print("Assistant:", response, "\n")



In [ ]:
TASK2_PROMPT = """You are a board-certified endocrinologist and patient-education specialist
writing for a hospital knowledge base used by both clinicians and patients.

Context: A user has asked a vague question ("Tell me about diabetes") and needs
a clear, medically accurate, well-structured explanation suitable for patient
education material reviewed by a doctor.

Constraints:
- Only include medically established, evidence-based information.
- Do NOT invent statistics, dosages, or drug names.
- If uncertain about a specific figure, say so instead of guessing.
- Keep each section concise (3-5 bullet points).
- Do not give a personal diagnosis; keep it general/educational.

Output strictly in Markdown with exactly these headings, in this order:
## Disease Overview
## Symptoms
## Risk Factors
## Prevention
## When to Consult a Doctor
"""

task2_response = generate_response(TASK2_PROMPT, temperature=0.2)
print(task2_response)

```markdown
# Diabetes

## Disease Overview
Diabetes is a chronic metabolic disorder characterized by high blood sugar levels over an extended period. It can be categorized into two main types: Type 1 and Type 2. Type 1 diabetes occurs when the body's immune system destroys insulin-producing beta cells in the pancreas, leading to absolute insulin deficiency. Type 2 diabetes develops due to insulin resistance combined with relative insulin deficiency, often associated with obesity and physical inactivity.

### Key Features
- **Symptoms**: Frequent urination, increased thirst, unexplained weight loss, fatigue, blurred vision, slow healing wounds, tingling or numbness in hands and feet.
- **Complications**: Nerve damage (neuropathy), kidney failure, heart disease, eye problems, foot infections, pregnancy complications.

## Symptoms
- **Common Signs**:
  - Increased thirst and frequent urination
  - Extreme hunger despite eating enough food
  - Unusual tiredness or weakness
  - Blurred vis

In [ ]:
TASK_FEWSHOT_PROMPT = """You convert short clinical notes into a structured summary.
Always follow the exact format shown in the examples: a "Symptoms" bulleted list,
then a "Severity" line with one word (Mild, Moderate, or Severe).

Example 1
Input: Patient has cough and fever.
Output:
Symptoms:
- Cough
- Fever
Severity:
Moderate

Example 2
Input: Patient reports mild headache and slight fatigue, no fever.
Output:
Symptoms:
- Headache
- Fatigue
Severity:
Mild

Example 3
Input: Patient has chest pain, shortness of breath, and oxygen saturation of 89%.
Output:
Symptoms:
- Chest pain
- Shortness of breath
- Low oxygen saturation (89%)
Severity:
Severe

Now convert the following note using the exact same format.
Input: Patient has fever for 3 days, dry cough, and body aches.
Output:
"""

task_response = generate_response(TASK3_FEWSHOT_PROMPT, temperature=0.1)
print(task3_response)

Symptoms:
- Fever for 3 days
- Dry cough
- Body aches
Severity:
Severe


In [ ]:
TASK_COT_PROMPT = """You are a clinical triage assistant. A patient presents with:
- Fever
- Cough
- Oxygen saturation (SpO2) = 88%

Think through this step by step internally: consider what an SpO2 of 88% indicates
(normal range is 95-100%), how it combines with fever and cough, and what urgency
level that implies. Do NOT show your step-by-step reasoning in the output.

After reasoning internally, respond with ONLY the following structured summary:

Urgency Level: <Low/Medium/High/Emergency>
Key Concern: <one short sentence>
Recommended Action: <one short sentence, e.g. seek immediate in-person care>

Do not include any other text before or after this block.
"""

task_response = generate_response(TASK4_COT_PROMPT, temperature=0.1)
print(task4_response)

Urgency Level: Medium
Key Concern: Possible respiratory compromise
Recommended Action: Monitor symptoms closely at home and contact healthcare provider if worsening.


In [ ]:
import json
import re

TASK5_JSON_PROMPT_TEMPLATE = """You are a clinical data extraction assistant.
Given the patient information below, respond with ONLY a single valid JSON object,
no explanation, no markdown code fences, no extra text before or after it.

The JSON must exactly match this schema:
{{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}}

Rules:
- "risk_level" must be one of: "Low", "Medium", "High".
- "symptoms" must be a JSON array of strings.
- Do not add extra keys. Do not omit any key.

Patient information: {patient_info}
"""

def get_structured_output(patient_info, retries=1):
    prompt = TASK5_JSON_PROMPT_TEMPLATE.format(patient_info=patient_info)
    raw = generate_response(prompt, temperature=0.0)

    def try_parse(text):
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return None
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    parsed = try_parse(raw)
    attempt = 0
    while parsed is None and attempt < retries:
        repair_prompt = prompt + "\n\nYour previous answer was not valid JSON. Reply again with ONLY the corrected valid JSON object."
        raw = generate_response(repair_prompt, temperature=0.0)
        parsed = try_parse(raw)
        attempt += 1

    return parsed, raw

parsed_json, raw_output = get_structured_output(
    "Patient has fever, dry cough, and sore throat for 2 days, no oxygen issues."
)
print("Raw model output:\n", raw_output)
print("\nParsed JSON:\n", parsed_json)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw model output:
 ```json
{
  "disease": "Influenza",
  "symptoms": [
    "fever",
    "dry cough",
    "sore throat"
  ],
  "risk_level": "Medium",
  "recommendation": "Stay home to avoid spreading the virus."
}
```

Parsed JSON:
 {'disease': 'Influenza', 'symptoms': ['fever', 'dry cough', 'sore throat'], 'risk_level': 'Medium', 'recommendation': 'Stay home to avoid spreading the virus.'}


In [ ]:
TASK_IMPROVED_PROMPT = """You are a clinical decision-support assistant. You must never say
"maybe" repeatedly without structure, and you must never guess without evidence.

A patient presents with the following symptoms: fever, fatigue, and body aches,
with recent travel to a malaria-endemic region.

Instructions:
- Internally consider the most likely differential diagnoses based on the symptoms given.
- Do NOT reveal your internal step-by-step reasoning.
- If key information is missing (e.g., travel history, test results) that would
  materially change the differential, explicitly ask for it instead of guessing.
- Present your answer as a ranked differential diagnosis list (most to least likely),
  each with a one-line justification tied to a specific symptom or fact given.
- End with a clear "Recommended Next Step" line.
- Never state a single definitive diagnosis without confirmatory tests; use
  probability language (e.g., "most likely", "possible") instead.

Output format:
## Differential Diagnosis
1. <condition> - <one-line justification>
2. <condition> - <one-line justification>
3. <condition> - <one-line justification>

## Recommended Next Step
<one or two sentences>
"""

task_response = generate_response(TASK7_IMPROVED_PROMPT, temperature=0.2)
print(task7_response)

## Differential Diagnosis
1. Malaria - Fever, fatigue, body aches, recent travel to a malaria-endemic region
2. Dengue Fever - Fever, headache, muscle pain, rash, recent travel to an area where dengue is common
3. Influenza - Fever, cough, sore throat, body aches, fatigue, recent exposure to respiratory viruses

## Recommended Next Step
If available, perform blood tests including complete blood count (CBC), peripheral smear examination for malaria parasites, and viral markers like IgM antibodies for influenza. Consider additional diagnostic tests if needed.
